In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(RAW_DATA_PATH / "olist_customers_dataset.csv")
orders = pd.read_csv(RAW_DATA_PATH / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DATA_PATH / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(RAW_DATA_PATH / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(RAW_DATA_PATH / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_DATA_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DATA_PATH / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    RAW_DATA_PATH / "product_category_name_translation.csv"
)

print("Datasets loaded.")

Datasets loaded.


In [10]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

print(orders[date_columns].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [11]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(f"{name:25} → {df.duplicated().sum():,} duplicate rows")

customers                 → 0 duplicate rows
orders                    → 0 duplicate rows
order_items               → 0 duplicate rows
order_payments            → 0 duplicate rows
order_reviews             → 0 duplicate rows
products                  → 0 duplicate rows
sellers                   → 0 duplicate rows
category_translation      → 0 duplicate rows


In [12]:
print("Unique customer IDs:", customers["customer_id"].nunique())
print("Unique customer identities:", customers["customer_unique_id"].nunique())
print("Unique orders:", orders["order_id"].nunique())
print("Unique products:", products["product_id"].nunique())
print("Unique sellers:", sellers["seller_id"].nunique())

Unique customer IDs: 99441
Unique customer identities: 96096
Unique orders: 99441
Unique products: 32951
Unique sellers: 3095


In [13]:
print(orders["order_status"].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [14]:
print("First purchase:", orders["order_purchase_timestamp"].min())
print("Last purchase:", orders["order_purchase_timestamp"].max())

First purchase: 2016-09-04 21:15:19
Last purchase: 2018-10-17 17:30:18


In [15]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

print("Total orders:", len(orders))
print("Delivered orders:", len(delivered_orders))
print("Excluded orders:", len(orders) - len(delivered_orders))

Total orders: 99441
Delivered orders: 96478
Excluded orders: 2963


In [16]:
print(
    delivered_orders[
        [
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ].isnull().sum()
)

order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


In [17]:
delivered_orders.to_csv(
    PROCESSED_DATA_PATH / "delivered_orders.csv",
    index=False
)

print("Saved:", PROCESSED_DATA_PATH / "delivered_orders.csv")

Saved: c:\Users\palla\Desktop\ecommerce-retention-intelligence\data\processed\delivered_orders.csv
